1. Cấu hình audit offline

Audit đọc artifact được Git theo dõi; clock lịch sử cố định, không xác nhận freshness hiện tại. Không gọi API, sửa catalogue hay chạy thuật toán GM-30.

In [ ]:
from pathlib import Path
import sys, json, copy
import pandas as pd
BASE = Path.cwd()
if not (BASE / 'scripts/validate_data_preparation.py').exists():
    BASE = BASE.parent if BASE.name == 'notebooks' else BASE / 'data-preparation'
assert (BASE / 'scripts/validate_data_preparation.py').exists(), 'Mở từ repo, data-preparation hoặc notebooks'
sys.path.insert(0, str(BASE / 'scripts'))
import validate_data_preparation as vd
REPORTS_DIR = BASE / 'datasets/data-validation/reports'
AUDIT_AT = vd.AUDIT_AT
print({'auditAt': AUDIT_AT, 'fixtureValidationAt': vd.ff.VALIDATION_AT, 'reports': str(REPORTS_DIR)})


2. Kiểm tra artifact và nguồn

Các nhóm kiểm độc lập ghi lỗi theo file/entity/dòng/trường; không coi dữ liệu draft thiếu thông tin là đã xác minh.

In [ ]:
report = vd.audit(at=AUDIT_AT)
display(pd.DataFrame(report['checks']))
display(pd.DataFrame(report['errors'], columns=['path','entity','row','field','message']))
display(pd.DataFrame.from_dict(report['inputFiles'], orient='index'))


3. Số liệu và thông tin còn thiếu

Mapping 219 tên, 39 món; giá là quan sát theo menu, không giá/người. Artwork null khi chưa đủ kiểm nội dung/license. Source rights unknown và reviewer null vẫn cần reviewer độc lập.

In [ ]:
display(pd.DataFrame([report['statistics']['mapping']]))
display(pd.DataFrame.from_dict(report['statistics']['editorial']['populated'], orient='index', columns=['populated']))
display(pd.DataFrame(report['sources']))
display(pd.DataFrame(report['warnings']))
print({'fixtures': report['statistics']['fixtures']['cases'], 'datasets': report['statistics']['fixtures']['datasets'], 'readyForPublish': report['readyForPublish']})


4. Minh họa lỗi trên bản sao mô phỏng

Chỉ đổi dữ liệu trong bộ nhớ để xem validator bắt ID trùng, FK taxonomy, giá âm/đơn vị sai. Expected GM-30 không được tính lại bằng thuật toán ở đây.

In [ ]:
invalid = copy.deepcopy(vd.bc.load_json(vd.ff.OUTPUT / 'base.json'))
invalid['entities']['dishes'].append(copy.deepcopy(invalid['entities']['dishes'][0]))
invalid['entities']['dishes'][0]['categoryIds'] = [vd.bc.stable_id('taxonomy','missing-notebook-demo')]
invalid['entities']['venueDishes'][0].update(priceMin=-1, unit='invented')
errors = vd.check_bundle(invalid, 'in-memory-demo.json', at=vd.ff.VALIDATION_AT)
assert errors
display(pd.DataFrame(errors))


5. Chạy lại và lưu báo cáo riêng

Tự tạo datasets/data-validation/reports, đọc lại report JSON và so hai lượt audit. Notebook executed do Python runner lưu tại đây; chưa kiểm trực tiếp Jupyter kernel. Không viết vào snapshot/fixture.

In [ ]:
assert report['validStructure'], report['errors']
again = vd.audit(at=AUDIT_AT)
assert again == report, 'Audit không ổn định hoặc input đã đổi'
vd.save_report(report, REPORTS_DIR)
assert json.loads((REPORTS_DIR / 'report.json').read_text()) == report
assert all(vd.ec.sha256(vd.ROOT / path) == meta['sha256'] for path,meta in report['inputFiles'].items())
stats = {'checks':len(report['checks']), 'errors':len(report['errors']), 'warnings':len(report['warnings']), 'networkCalls':0, 'eligibilityExecuted':False}
print(stats)
print(REPORTS_DIR / 'REPORT.md')
